### Libraries

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt



from sklearn.impute import SimpleImputer

def load_data():
    candidates = [
        Path('/content/dataset_limpio_ok.csv'),  
        Path('data/raw/dataset_limpio_ok.csv'),   
        Path('dataset_limpio_ok.csv'),            
    ]
    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            print("=== Dataset cargado ===")
            print(f"Archivo: {p} | Filas={df.shape[0]} | Columnas={df.shape[1]}")
            display(df.head(3))
            return df
    raise FileNotFoundError('No se encontró dataset_limpio_ok.csv en rutas conocidas.')

#loaded ds
df = load_data()  # Llama a la función y guarda el DataFrame en 'df'

=== Dataset cargado ===
Archivo: /content/dataset_limpio_ok.csv | Filas=235 | Columnas=36


,profesionales,edad,motivo_de_consulta,medio_por_el_que_ingresa,genero,nacionalidad,barrio,municipio,localidad,estado_civil,...,simbolica,ambiental,politica,digital,cant_tipos_violencias_por_persona,denuncio,medidas_de_proteccion,fecha_fin_de_vigencia,personas_a_cargo,red_vincular
0,fanny___dana,28.0,violencia,desconocido,mujer,argentina,bordeu,bahia_blanca,bahia_blanca,soltera,...,0,0,0,0,2.0,no,no,NaN,no,parientes_convivientes
1,agus__analé,43.0,violencia,desconocido,mujer,argentina,pacifico,bahia_blanca,bahia_blanca,desconocido,...,0,1,0,0,4.0,si,prohibición_de_acercamiento,NaN,hija_hijo,parientes_no_convivientes
2,fanny__majo,53.0,violencia,desconocido,mujer,argentina,pampa_central,bahia_blanca,bahia_blanca,soltera,...,0,0,0,0,2.0,no,no,NaN,no,parientes_no_convivientes


In [ ]:
import unicodedata
import re

# Universidad Nacional de la Matanza

Specialization in Data Science

Benko Teo
Cura Diego
Riganti Valentina
Sanjuan Oriana.

Model based on data from the Dirección General de Género de Bahía Blanca

# **Stage: Feature Engineering**

### Consultation variables (assigned professional, reason for consultation, and intake channel):

The variables professionals (profesionales) and reason for consultation (motivo_de_consulta) must be dropped for strategic reasons (high cardinality and low predictive value, respectively):

1. Drop 'profesionales' due to Extreme Cardinality (OHE cannot be used).
2. Drop 'motivo_de_consulta' due to Bias/Low Variability (almost 86% is 'violencia').

In [ ]:
cols_to_engineer = ['profesionales', 'motivo_de_consulta']

df = df.drop(columns=cols_to_engineer, errors='ignore')

# Verificación de la Ingeniería
print("--- Feature Engineering: Variable Elimination ---")
print(f"Strategically dropped columns: {cols_to_engineer}")
print(f"Remaining columns: {df.shape[1]}")

--- Ingeniería de Características: Eliminación de Variables ---
Columnas eliminadas por estrategia: ['profesionales', 'motivo_de_consulta']
Total de columnas restantes: 34


The intake channel variable (medio_por_el_que_ingresa) has high cardinality (10 categories) but many have low frequency (less than 3% of the data).

Decision: functionally group the small categories to improve model stability without losing essential information about the consultation channel.

The grouping of the original channels results in:


| Functional Category | Original Categories | Observation |
| --- | --- | --- |
| Judicial Channel (CANAL_JUDICIAL) | judicial notification (notificación_judicial) | Represents the most formal/mandatory intake channel|
| Spontaneous Channel (CANAL_ESPONTANEO) | spontaneous (espontánea) | High predictive value (victim's initiative). |
| Third-party Channel (CANAL_TERCERO) | other (otra) | Captures direct non-institutional referrals. |
| Assistance Channel (CANAL_ASISTENCIAL) | scheduled appointment (turno_programado), police station (comisaría), health unit (unidad_sanitaria), social organization... (organización_social...), educational institution (institución_educativa), hospital (hospital)| AStrategic Grouping: the victim was referred or assisted by another entity (health, police, social). |
| Unknown (DESCONOCIDO) | unknown (desconocido) | Preserves the missing data information. |

In [ ]:
columna = 'medio_por_el_que_ingresa'
nuevo_nombre = 'canal_ingreso_agrupado'

# Grouping
df[nuevo_nombre] = np.select(
    [
        # 1. CANAL JUDICIAL
        df[columna].str.contains('notificación_judicial', na=False),

        # 2. CANAL ESPONTANEO
        df[columna] == 'espontánea',

        # 3. CANAL TERCERO
        df[columna] == 'otra',

        # 4. CANAL ASISTENCIAL (Agrupa todos los de baja frecuencia/instituciones)
        df[columna].isin(['turno_programado', 'comisaría', 'unidad_sanitaria', 'organización_social_institución_comunitaria', 'institución_educativa', 'hospital']),

        # 5. DESCONOCIDO
        df[columna] == 'desconocido'
    ],
    [
        'CANAL_JUDICIAL',
        'CANAL_ESPONTANEO',
        'CANAL_TERCERO',
        'CANAL_ASISTENCIAL',
        'DESCONOCIDO'
    ],
    default='OTROS_A_REVISAR' # Categoría de seguridad para cualquier valor no mapeado
)

# Codification OHE
df = pd.get_dummies(df, columns=[nuevo_nombre], prefix='INGR', drop_first=False)

# Remove original column
df = df.drop(columns=[columna])

# Verification
print("\n--- Verification of 'medio_por_el_que_ingresa' ---")
print(df[[col for col in df.columns if col.startswith('INGR_')]].sum().to_frame(name='Conteo Final').to_markdown())


--- Verificación del Agrupamiento de 'medio_por_el_que_ingresa' ---
|                        |   Conteo Final |
|:-----------------------|---------------:|
| INGR_CANAL_ASISTENCIAL |             28 |
| INGR_CANAL_ESPONTANEO  |             50 |
| INGR_CANAL_JUDICIAL    |             83 |
| INGR_CANAL_TERCERO     |             26 |
| INGR_DESCONOCIDO       |             48 |


### Location variables of the reporting person:

The variables municipality (municipio) and locality (localidad) provide redundant information and have little predictive value, given that over 95% of the cases occur in Bahía Blanca.

1. Drop municipality (municipio). After cleaning, 98.3% of the cases are Bahía Blanca (bahía_blanca). It is an almost constant feature and adds no predictive value to the model.

2. Drop locality (localidad). After cleaning, 97% of the cases are Bahía Blanca (bahía_blanca). It is an almost constant feature and is redundant with municipality (municipio) (already dropped).

In [ ]:
df = df.drop(columns=['municipio', 'localidad'], errors='ignore')

The nationality variable (nacionalidad) is heavily skewed towards Argentine (argentina) (93.19%). To convert it into a useful feature, a simple binary grouping will be applied. This avoids biases from the majority class and preserves information about nationalities other than Argentine.

In [ ]:
col_nac = 'nacionalidad'
df['nacionalidad_agrupada'] = np.where(
    df[col_nac] == 'argentina',
    'ARGENTINA',
    'NO_ARGENTINA'
)

print("--- Verification of nationality---")
print("\nGrouped Nationality Value Counts")
print(df[[col for col in df.columns if col.startswith('NAC_')]].sum().to_frame(name='Conteo').to_markdown())

--- Verificación de la Ingeniería de Ubicación ---

Conteo de Nacionalidad Agrupada:
| Conteo   |
|----------|


In [ ]:
df = df.drop(columns=[col_nac], errors='ignore')

For the neighborhood variable (barrio), geographic clustering (zoning) will be applied to reduce the almost 100 categories down to 7 zones. This approach preserves the spatial information.

In [ ]:
# Neighborhood Processing (Geographic Zoning)
# Normalized classification lists

zona_centro_norm = [
    'centro', 'microcentro', 'naposta', 'pacifico', 'b pacifico', 'pedro pico', 'ricchieri'
]

zona_norte_norm = [
    'agua blanca', 'altos de bahia', 'altos de la carrindanga', 'jardin del este', 'la falda',
    'los alamos', 'millamapu', 'molina campos', 'palihue', 'b patagonia', 'patagonia norte',
    'parque del sol', 'parque norte', 'parque patagonia', 'parque sesquicentenario',
    'portal del este', 'san agustin', 'san ignacio', 'santa margarita', 'solares norte', 'universitario'
]

zona_villas_norm = [
    '1ro de mayo', '5 de abril', '27 de junio', 'bap', 'comercial', 'evita', 'barrio evita',
    'rosendo lopez', 'sutiaga', 'uom', 'upcn', 'villa amaducci', 'villa belgrano',
    'villa cerrito', 'villa floresta', 'villa libre', 'villa mitre', 'villa gral mitre',
    'villa rosario', 'villa rosas', 'villa serra', 'villa soldati'
]

zona_noroeste_norm = [
    'aldea romana', 'altos del sur', 'ate', 'ate 1', 'ate 2', 'ate 3', 'ate 4', 'ate 5',
    'avellaneda', 'bancario', 'bellavista', 'cgt', 'campana del desierto', 'cooperacion 2',
    'coronel maldonado', 'don bosco', 'don ramiro', 'el bosque', 'el matadero', 'el nacional',
    'el prado', 'empl de comercio', 'floresta', 'fonavi', '9 de noviembre', 'la canada',
    'las canitas', 'latino', 'loma alta', 'loma paraguaya', 'los chanares', 'lujan',
    'luz y fuerza', 'malvinas argentinas', 'mapuche', 'mataderos', 'nor oeste', 'nueva central',
    'palos verdes', 'pampa central', 'piedra buena', 'polo', 'prensa', 'suem',
    'san cayetano', 'san francisco', 'san jorge', 'san martin', 'san miguel', 'santa catalina',
    'santa teresita', 'spurr', 'stella maris', 'thompson', 'viajantes del sur', 'villa alegre',
    'villa delfina', 'villa don bosco', 'villa duprat', 'villa elena', 'villa esperan',
    'villa gloria', 'villa hipodromo', 'villa irupe', 'villa italia', 'villa loreto',
    'villa moresino', 'villa muniz', 'villa nocito', 'villa nueva', 'villa ressia',
    'villa rica', 'villa roma', 'villa sanchez', 'vista alegre'
]

zona_exteriores_norm = [
    '17 de mayo', '26 de septiembre', 'aeropuerto', 'villa aeropuerto', 'amef',
    'boulevard white', 'cnel falcon', 'essa', 'essa 2', 'espora', 'general cerri',
    'grunbein', 'harding green', 'villa harding green', 'ingeniero white', 'kilometro 5',
    'obrero', 'parque industrial', 'pescadores', 'petroquimico', 'playa serena', 'saladero',
    'supe', 'villa bordeu', 'villa gral arias'
]

# Main mapping function

def clasificar_zona(barrio):
    """
    Process the PRE-CLEANED neighborhood to match the zoning list format (including spaces).    
    """
    # Standarization
    barrio_norm = str(barrio).strip().lower().replace('_', ' ')

    # 1. Identify null values or explicit 'missing data' (sin datos) entries.
    # (If normalization returns an empty string, the original value was null or nan)
    if barrio_norm == 'desconocida':
        return 'DESCONOCIDO' 

    # 2. Validate primary zones by comparing the adapted format against the zoning lists.
    # NOTE: The normalized neighborhood variable (barrio_norm) is used directly for matching since it has been formatted with spaces.

    if barrio_norm in zona_centro_norm:
        return 'CENTRO_Y_MACROCENTRO'
    if barrio_norm in zona_norte_norm:
        return 'NORTE'
    if barrio_norm in zona_villas_norm:
        return 'SUR'
    if barrio_norm in zona_noroeste_norm:
        return 'NOROESTE'
    if barrio_norm in zona_exteriores_norm:
        return 'INGENIERO_WHITE'

    # 3. Check for partial cases or sub-neighborhoods (e.g., fonavi i, fonavi ii).
    
    if 'fonavi' in barrio_norm:
        return 'NOROESTE'
    if 'ate' in barrio_norm:
        return 'NOROESTE'
    if 'harding green' in barrio_norm:
        return 'INGENIERO_WHITE'
    if 'bordeu' in barrio_norm:
        return 'NORTE'

    # 4. If none of the above matches, it is classified as 'others' (otros).
    return 'OTROS'

# aplication

df['ZONA'] = df['barrio'].apply(clasificar_zona)

In [ ]:
print(df['ZONA'].value_counts())

ZONA
OTROS                   84
NOROESTE                57
SUR                     29
DESCONOCIDO             20
INGENIERO_WHITE         20
NORTE                   14
CENTRO_Y_MACROCENTRO    11
Name: count, dtype: int64


In [ ]:
# Removal of the original column
df = df.drop(columns=['barrio'], errors='ignore')

In [ ]:
# One-Hot Encoding (OHE)
ohe_targets = ['nacionalidad_agrupada', 'ZONA']
df = pd.get_dummies(df, columns=ohe_targets, prefix=['NAC', 'BA'], drop_first=False)


# 5. Verification
print("--- Value Counts for Zoning (barrio_zonificado) ---")

print("\nConteo de Barrio Zonificado:")
print(df[[col for col in df.columns if col.startswith('BA_')]].sum().to_frame(name='Count').to_markdown())

--- Verificación de la Ingeniería de Ubicación ---

Conteo de Barrio Zonificado:
|                         |   Conteo |
|:------------------------|---------:|
| BA_CENTRO_Y_MACROCENTRO |       11 |
| BA_DESCONOCIDO          |       20 |
| BA_INGENIERO_WHITE      |       20 |
| BA_NOROESTE             |       57 |
| BA_NORTE                |       14 |
| BA_OTROS                |       84 |
| BA_SUR                  |       29 |


### Variables for personal attributes:

A binary feature is generated for gender identification. It groups the minority ('other' [otro]) and unknown classes into a single category: 'NOT_WOMAN' (NO_MUJER).

This approach accounts for all cases where the individual inquiring is not identified as a woman.

In [ ]:
# Feature Engineering for Gender (genero) (Binary Grouping)
col_genero = 'genero'
grouped_col_genero = 'genero_grouped'

# Implement binary classification logic (Woman vs. All other categories).
df[grouped_col_genero] = np.where(
    df[col_genero] == 'mujer',
    'MUJER',
    'NO_MUJER'
)

# Implement One-Hot Encoding (OHE) on the new feature 
# (this results in 2 columns; although one could be removed using drop_first=True, we will retain both for the time being).
df = pd.get_dummies(df, columns=[grouped_col_genero], prefix='GEN', drop_first=False)


# verification
print("--- Age Feature Engineering Check ---")
genero_cols = [col for col in df.columns if col.startswith('GEN_')]
print("\nValue Counts for Grouped Gender (GEN):")
print(df[genero_cols].sum().to_frame(name='Count').to_markdown())

--- Verificación de la Ingeniería de Características (Edad) ---

Conteo de Género Agrupado (GEN):
|              |   Conteo |
|:-------------|---------:|
| GEN_MUJER    |      228 |
| GEN_NO_MUJER |        7 |


In [ ]:
# Drop original column
df = df.drop(columns=[col_genero], errors='ignore')

Age groups were defined to discretize the continuous numerical variable 'age' (edad). For categorical stability, the intervals were determined using quartiles ($Q1=29.5$, Median=36, $Q3=45$) and the maximum observed age ($85$).

In [ ]:
# Feature Engineering for Age (edad) (Age Binning)

col_edad = 'edad'
binned_col_edad = 'rango_etario'

# Establish numerical thresholds
bins = [17, 29.5, 36, 45, 85]
labels = ['17_a_29', '30_a_36', '37_a_45', '46_a_85'] 

# Generate discretized features using pd.cut
df[binned_col_edad] = pd.cut(
    df[col_edad],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

# Implement One-Hot Encoding (OHE) on the newly created age range feature.
df = pd.get_dummies(df, columns=[binned_col_edad], prefix='EDAD', drop_first=False)


# verification
print("--- Age Feature Engineering Check ---")

edad_cols = [col for col in df.columns if col.startswith('EDAD_')]
print("\nValue Counts for Age Bins (EDAD):")
print(df[edad_cols].sum().to_frame(name='Count').to_markdown())

print(f"\nData type verification for 'age' (edad) (should be float/int): {df['edad'].dtype}")


--- Verificación de la Ingeniería de Características (Edad) ---

Conteo de Rangos Etarios (EDAD):
|              |   Conteo |
|:-------------|---------:|
| EDAD_17_a_29 |       59 |
| EDAD_30_a_36 |       60 |
| EDAD_37_a_45 |       51 |
| EDAD_46_a_85 |       55 |

Tipo de dato de 'edad' (debe ser float/int): float64


In [ ]:
# 2.4. Delete intermediate range and age columns.
df = df.drop(columns=[binned_col_edad], errors='ignore')
df = df.drop(columns=['edad'], errors='ignore')

### Socioeconomic and individual status features:

Regarding the 'marital status' (estado_civil) variable, a binary classification was implemented to reflect distinct relational dynamics and their associated risks.

- Married/Partnered: Includes legal marriage and registered cohabitation. From a functional perspective, these indicate stable bonds with shared responsibilities.

- Separated/Divorced: These categories mark the dissolution of a formal union, a factor often associated with increased vulnerability and risk of post-breakup violence.

In [ ]:
# --- 1. Feature Engineering for Marital Status (estado_civil) ---
col_ec = 'estado_civil'
# 1.1. Mapping Cohabitation (Unión Convivencial) to the Married (Casada) category
df[col_ec] = df[col_ec].replace('unión convivencial', 'casada')
# 1.2. Merging Separated and Divorced categories
df[col_ec] = df[col_ec].replace({'separada': 'separada_divorciada', 'divorciada': 'separada_divorciada'})
# 1.3. Implementation of One-Hot Encoding (OHE) on the processed feature.
df = pd.get_dummies(df, columns=[col_ec], prefix='EC', drop_first=False)
df = df.drop(columns=['estado_civil'], errors='ignore')

Regarding 'educational level' (nivel educativo), both the current level (Secondary, Tertiary, Primary) and the educational status (incomplete, ongoing, complete) are specified.

Therefore, grouping these detailed levels into broader categories (Primary, Secondary, Superior) was considered the best approach to reduce cardinality (from 14 to 4) and create a more stable feature for measuring socioeconomic status and resources.

In [ ]:
# --- 2. Feature Engineering for Educational Level (nivel_educativo)---
col_ne = 'nivel_educativo'
mapping_ne = {
    # Primario
    'primario incompleto': 'PRIMARIO', 'primario completo': 'PRIMARIO',
    'sin estudios/ sabe leer y escribir': 'OTROS', 'escuela especial completa': 'OTROS',
    # Secundario
    'secundario incompleto': 'SECUNDARIO', 'secundario completo': 'SECUNDARIO', 'secundario en curso': 'SECUNDARIO',
    # Superior
    'terciario incompleto': 'SUPERIOR', 'terciario completo': 'SUPERIOR', 'terciario en curso': 'SUPERIOR',
    'universitario incompleto': 'SUPERIOR', 'universitario completo': 'SUPERIOR', 'universitario en curso': 'SUPERIOR',
    'desconocido': 'DESCONOCIDO'
}
df['nivel_educativo_grouped'] = df[col_ne].replace(mapping_ne)
df = pd.get_dummies(df, columns=['nivel_educativo_grouped'], prefix='NE', drop_first=False)
df = df.drop(columns=['nivel_educativo'], errors='ignore')

The 'employment status' (situacion laboral) variable has many low-frequency categories.
Risk assessment should focus on economic instability or the dependency of the person seeking consultation.
A decision was made to group by Income Type/Dependency into $\approx 4$ key categories:
* UNEMPLOYED (DESEMPLEO): Includes 'does not work' (no trabaja), 'homemaker' (ama de casa), and 'unemployed' (desocupade).
* FORMAL: Formal employment.
* INFORMAL: Includes 'self-employed' (cuenta propista) and 'informal work' (trabajo informal).
* RETIRED/PENSIONER (JUB/PEN): Includes 'retired' (jubilade) and 'pensioner' (pensionada).

In [ ]:
# --- 3. Feature Engineering for Employment Status (situacion_laboral) ---
col_sl = 'situacion_laboral'
mapping_sl = {
    'no trabaja': 'DESEMPLEO', 'desocupade': 'DESEMPLEO', 'ama de casa': 'DESEMPLEO', # Desempleo
    'trabajo formal': 'FORMAL',
    # Informal/Cuenta Propista
    'cuenta propista no registrado': 'INFORMAL', 'cuenta propista registrado': 'INFORMAL',
    'trabajo informal': 'INFORMAL', 'esporádico informal': 'INFORMAL', 'esporádico formal': 'INFORMAL',
    # Jubilación/Pensión
    'jubilade': 'JUB_PEN', 'pensionada': 'JUB_PEN',
    'desconocido': 'DESCONOCIDO'
}
df['situacion_laboral_grouped'] = df[col_sl].replace(mapping_sl)
df = pd.get_dummies(df, columns=['situacion_laboral_grouped'], prefix='SL', drop_first=False)
df = df.drop(columns=['situacion_laboral'], errors='ignore')

For the 'receives_state_benefit' (percibe_prestacion_estatal) variable, the key question is: Is there formal State support (AUH/SUAF/Pension)?

This allowed for grouping into 3 functional categories:

* KEY_BENEFIT (BENEFICIO_CLAVE): (AUH/SUAF) --> YES

* DOES_NOT_RECEIVE (NO_PERCIBE): --> NO

* OTHER_BENEFITS (OTROS_BENEFICIOS): (pensions, etc.) --> YES

A binary grouping is performed to answer the question with Yes = 1 / No = 0.

In [ ]:
# --- 4. Feature Engineering for State Benefits (percibe_prestacion_estatal) ---
col_ppe = 'percibe_prestacion_estatal'
df['prestacion_grouped'] = np.select(
    [
        df[col_ppe].str.contains('auh') | df[col_ppe].str.contains('suaf'),
        df[col_ppe].str.contains('no percibe'),
        df[col_ppe].str.contains('desconocido')
    ],
    ['BENEFICIO_CLAVE', 'NO_PERCIBE', 'DESCONOCIDO'],
    default='OTROS_BENEFICIOS' # includes others 
)
df = pd.get_dummies(df, columns=['prestacion_grouped'], prefix='PPE', drop_first=False)
df = df.drop(columns=[col_ppe], errors='ignore')

To group the 'housing' (vivienda) variable, the most significant risk factor identified was household dependency or instability.

It was grouped into 4 categories:

* OWNED (PROPIA): Maximum stability.

* RENTED (ALQUILADA): High fixed cost.

* BORROWED/SHARED (PRESTADA/COMPARTIDA): High dependency.

* AT RISK/OTHER (RIESGO/OTROS): Homelessness, occupied/squatted.

In [ ]:
# --- 5. Feature Engineering for Housing ---
col_viv = 'vivienda'
mapping_viv = {
    'propia': 'PROPIA', 'alquilada': 'ALQUILADA',
    'prestada': 'PRESTADA_COMPARTIDA', 'compartida': 'PRESTADA_COMPARTIDA',
    'desconocido': 'DESCONOCIDO',
    # Group in RIESGO_ALTO
    'situación de calle': 'RIESGO_ALTO', 'ocupada': 'RIESGO_ALTO',
    'propiedad bien ganancial': 'RIESGO_ALTO', 'vivienda del pea': 'RIESGO_ALTO'
}
df['vivienda_grouped'] = df[col_viv].replace(mapping_viv)
df = pd.get_dummies(df, columns=['vivienda_grouped'], prefix='VIV', drop_first=False)
df = df.drop(columns=[col_viv], errors='ignore')

For the 'health insurance' (obra social) variable, the key question is: Does it exist?

This allows for binarization with a logical mapping: $1$ represents the presence of the attribute --> $1 = \text{YES}$ and $0 = \text{NO/UNKNOWN}$.

In [ ]:
# --- 6. Feature Engineering for Health Coverage (obra_social)  ---
col_os = 'obra_social'

df['obra_social_bin'] = np.where(
    df[col_os] == 'si',
    1, # has health coverage
    0  # doesn't have or it is unknown
)

In [ ]:
# Remove original column
df = df.drop(columns=[col_os], errors='ignore')

In [ ]:
# --- verification ---
print("--- Socioeconomic Feature Check ---")
print(f"Total features (Post-OHE):{df.shape[1]}")

# Health coverage
print("\nValue Counts for Health Insurance (Binary, 1=SI):")
print(df['obra_social_bin'].value_counts().to_frame(name='Count').to_markdown())

# Education 
ne_cols = [col for col in df.columns if col.startswith('NE_')]
print("\nValue Counts for Consolidates Educational Levels:")
print(df[ne_cols].sum().to_frame(name='Count').to_markdown())

# 3. employment
sl_cols = [col for col in df.columns if col.startswith('SL_')]
print("\nValue Counts for Consolidated Employment Status:")
print(df[sl_cols].sum().to_frame(name='Count').to_markdown())

# marital status
ec_cols = [col for col in df.columns if col.startswith('EC_')]
print("\nDistribution of Marital Status Categories:")
print(df[ec_cols].sum().to_frame(name='Count').to_markdown())

# housing
ec_cols = [col for col in df.columns if col.startswith('VIV_')]
print("\nDistribution of Residential Stability (Housing):")
print(df[ec_cols].sum().to_frame(name='Count').to_markdown())

# state benefits
ec_cols = [col for col in df.columns if col.startswith('PPE_')]
print("\nGrouped State Benefit Count:")
print(df[ec_cols].sum().to_frame(name='Count').to_markdown())

--- Verificación de la Ingeniería de Características Socioeconómicas ---
Total de columnas después del OHE: 66

Conteo de Obra Social (Binario, 1=SI):
|   obra_social_bin |   Conteo |
|------------------:|---------:|
|                 0 |      174 |
|                 1 |       61 |

Conteo de Nivel Educativo Agrupado (NE):
|                |   Conteo |
|:---------------|---------:|
| NE_DESCONOCIDO |       30 |
| NE_OTROS       |        5 |
| NE_PRIMARIO    |       29 |
| NE_SECUNDARIO  |      112 |
| NE_SUPERIOR    |       59 |

Conteo de Situación Laboral Agrupada (SL):
|                |   Conteo |
|:---------------|---------:|
| SL_DESCONOCIDO |       48 |
| SL_DESEMPLEO   |       69 |
| SL_FORMAL      |       38 |
| SL_INFORMAL    |       70 |
| SL_JUB_PEN     |       10 |

Conteo de Estado Civil Agrupado (EC):
|                        |   Conteo |
|:-----------------------|---------:|
| EC_casada              |       63 |
| EC_desconocido         |       25 |
| EC_separada_divorc

### Descriptors for Clinical, Risk, and Household Features:

Methodological Decisions:

1. Clinical Variables (Diagnosis, Treatment, CUD): Due to the high prevalence of 'unknown' values ($51\%$–$60\%$), these were recoded as 'NO' following institutional guidelines. Regarding the diagnosis variable, low-frequency categories (e.g., chronic illness, antidepressant use) were consolidated into a single 'YES' category to capture the presence of any clinical risk factor.

2. Family/Perpetrator Variables: The features hijos_pea and convivencia_pea are already binary and require only numerical encoding ($1/0$) for model compatibility.

In [ ]:

cols_to_engineer = ['diagnostico', 'tratamiento', 'posee_cud', 'hijos_pea', 'convivencia_pea']

# --- 1. Feature Engineering: 'diagnostico' Consolidation ---
col_diag = 'diagnostico'
# 1.1. Grouping clinical minorities into 'YES'
df[col_diag] = df[col_diag].replace({
    'consum antidep.': 'si',
    'enfermedad crónica': 'si'
})
# 1.2. Mapping 'unknown' values to 'no'.
df[col_diag] = df[col_diag].replace({'desconocido': 'no'})
# 1.3. Binary Encoding: 1=YES / 0=NO
df[f'{col_diag}_bin'] = df[col_diag].replace({'si': 1, 'no': 0})


# --- 2. Feature Engineering: 'tratamiento' and 'cud' Consolidation ---
for col in ['tratamiento', 'posee_cud']:
    # 2.1. Mapping 'unknown' values to 'no'.
    df[col] = df[col].replace({'desconocido': 'no'})
    # 2.2.Binary Encoding: 1=YES / 0=NO
    df[f'{col}_bin'] = df[col].replace({'si': 1, 'no': 0})


# --- 3. FAMILY VARIABLES FEATURE ENGINEERING ---
for col in ['hijos_pea', 'convivencia_pea']:
        df[f'{col}_bin'] = df[col].replace({'si': 1, 'no': 0})


# --- 4. Verification ---
print("--- Verificación Final de la Ingeniería (Clínicas y Riesgo) ---")

# Verification of 'diagnostico'
print("\nValue Counts for Binarized 'diagnostico' (DIAG_bin: 1=SI):")
print(df['diagnostico_bin'].value_counts().to_frame(name='Count').to_markdown())

# Verification of 'CUD'
print("\nBinary Disability Certificate (CUD) Count (posee_cud_bin: 1=SI):")
print(df['posee_cud_bin'].value_counts().to_frame(name='Count').to_markdown())

# Verification of 'hijos_pea'
print("\nBinary Count of Children with Perpetrator (hijos_pea) (hijos_pea_bin: 1=SI):")
print(df['hijos_pea_bin'].value_counts().to_frame(name='Count').to_markdown())

--- Verificación Final de la Ingeniería (Clínicas y Riesgo) ---

Conteo de Diagnóstico Binario (DIAG_bin: 1=SI):
|   diagnostico_bin |   Conteo |
|------------------:|---------:|
|                 0 |      193 |
|                 1 |       42 |

Conteo de Posesión de CUD Binario (posee_cud_bin: 1=SI):
|   posee_cud_bin |   Conteo |
|----------------:|---------:|
|               0 |      219 |
|               1 |       16 |

Conteo de Hijos PEA Binario (hijos_pea_bin: 1=SI):
|   hijos_pea_bin |   Conteo |
|----------------:|---------:|
|               0 |      131 |
|               1 |      104 |


/tmp/ipython-input-431716.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'{col_diag}_bin'] = df[col_diag].replace({'si': 1, 'no': 0})
/tmp/ipython-input-431716.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'{col}_bin'] = df[col].replace({'si': 1, 'no': 0})
/tmp/ipython-input-431716.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd

In [ ]:
# Verificatinon of 'convivencia_PEA'
print("\nValue Counts for Cohabitation with Perpetrator (convivencia_pea_bin: 1=SI):")
print(df['convivencia_pea_bin'].value_counts().to_frame(name='Count').to_markdown())


Conteo de Convivencia PEA Binario (convivencia_pea_bin: 1=SI):
|   convivencia_pea_bin |   Conteo |
|----------------------:|---------:|
|                     0 |      218 |
|                     1 |       17 |


In [ ]:
# Verification of 'tratamiento' (TRA)
print("\nBinary Treatment Count (TRA_bin: 1=SI):")
print(df['tratamiento_bin'].value_counts().to_frame(name='Count').to_markdown())


Conteo de Tratamiento Binario (TRA_bin: 1=SI):
|   tratamiento_bin |   Conteo |
|------------------:|---------:|
|                 0 |      202 |
|                 1 |       33 |


In [ ]:
#Drop original columns
df = df.drop(columns=['diagnostico', 'tratamiento', 'posee_cud', 'hijos_pea', 'convivencia_pea'], errors='ignore')

### Violence indicators

The 'number of dependents' (personas_a_cargo) variable is a count variable, but the frequencies for higher values ($5, 6, 7$) are extremely low ($<2\%$).

This grouping stabilizes the model and improves interpretability by clustering cases into functional risk ranges:

- [CERO] ZERO ($0$): Absence of family-related risk.
-  [BAJO_RIESGO] LOW_RISK ($1, 2$)
- [MEDIO_RIESGO] MEDIUM_RISK ($3, 4$)
- [ALTO_RIESGO] HIGH_RISK ($\geq 5$)

In [ ]:
#  Aggregation Logic

col_count = 'cant_personas_a_cargo'
col_grouped = 'personas_cargo_grouped'

# --- 1. Cleaning ---
# Null/No -> 0 
df[col_count] = df[col_count].replace('NO', '0')
df[col_count] = pd.to_numeric(df[col_count], errors='coerce').fillna(0).astype(int)

# --- 2. APPLYING FUNCTIONAL GROUPING ---
df[col_grouped] = np.select(
    [
        df[col_count] == 0,
        df[col_count].isin([1, 2]),
        df[col_count].isin([3, 4]),
        df[col_count] >= 5
    ],
    [
        'CERO',
        'BAJO_RIESGO',
        'MEDIO_RIESGO',
        'ALTO_RIESGO'
    ],
    default='ERROR_LOGICO' # for safety

# --- 3. verification ---
verification_counts = df[col_grouped].value_counts().to_frame(name='Count')
total_registros = len(df)

print("--- Validation of Engineered Features in 'cant_personas_a_cargo' (Agrupamiento) ---")
print(f"Total Count of Entries: {total_registros}")
print("\nValue Counts per Functional Group:")
print(verification_counts.to_markdown())

--- Verificación de la Ingeniería de 'cant_personas_a_cargo' (Agrupamiento) ---
Total de Registros: 235

Conteo por Categoría Funcional:
| personas_cargo_grouped   |   Conteo |
|:-------------------------|---------:|
| CERO                     |       93 |
| BAJO_RIESGO              |       89 |
| MEDIO_RIESGO             |       49 |
| ALTO_RIESGO              |        4 |


In [ ]:
# Drop original column
df = df.drop(columns=[col_count], errors='ignore')

The feature modalidad_de_violencia exhibits a highly skewed distribution, where 'domestic' accounts for $80\%$ of the observations. Given its near-zero variance and minimal information gain, this variable was excluded to prevent model overfitting and improve classification efficiency.

In [ ]:
col_modalidad = 'modalidad_de_violencia'

#drop
df = df.drop(columns=[col_modalidad], errors='ignore')

print(f"Column '{col_modalidad}' dropped for low variance (80% is 'doméstica').")

Columna 'modalidad_de_violencia' eliminada por baja variabilidad (80% es 'doméstica').


In gender-based violence research, risk accumulation is a robust predictor of critical outcomes.

Rather than analyzing individual violence types in isolation, the composite feature SUMA_VIOLENCIA was developed. This approach better captures case complexity by quantifying the co-occurrence and overlap of multiple forms of violence.

In [ ]:
# types of violence (list)

violence_cols = [
    'fisica', 'psicologica', 'sexual', 'economica',
    'simbolica', 'ambiental', 'politica', 'digital'
]
original_count_col = 'cant_tipos_violencias_por_persona'

# --- 1. creation of SUMA (acumulation of risk) ---

# checking int64 for sum
for col in violence_cols:
    
    df[col] = df[col].astype(int)

# create feature suma
df['SUMA_VIOLENCIA'] = df[violence_cols].sum(axis=1).astype(int)

# 3. verification
print("--- Feature Engineering Verification (Violencia) ---")

print("\n--- Count of new Feature 'SUMA_VIOLENCIA' ---")
print(df['SUMA_VIOLENCIA'].value_counts().sort_index().to_frame(name='Count').to_markdown())

print(f"\nType of 'SUMA_VIOLENCIA': {df['SUMA_VIOLENCIA'].dtype}")

--- Verificación de la Ingeniería de Características (Violencia) ---

--- Conteo del Nuevo Feature 'SUMA_VIOLENCIA' ---
|   SUMA_VIOLENCIA |   Conteo |
|-----------------:|---------:|
|                0 |       60 |
|                1 |       40 |
|                2 |       51 |
|                3 |       36 |
|                4 |       28 |
|                5 |       13 |
|                6 |        6 |
|                7 |        1 |

Tipo de dato de 'SUMA_VIOLENCIA': int64


In [ ]:
# Drop original column
df = df.drop(columns=[original_count_col], errors='ignore')

print(f"La columna '{original_count_col}' ha sido eliminada.")

La columna 'cant_tipos_violencias_por_persona' ha sido eliminada.


### Socio-Demographic Indicators

For the 'number of dependents' (personas_a_cargo) variable, a functional grouping of 'none' + 'others' $\rightarrow$ NO_OTHERS allows for the consolidation of cases with no children under care and very low-frequency categories.

Regarding the 'support network' (red_vincular), a grouping of 'none' + 'neighbors' $\rightarrow$ RV_NO_SUPPORT was created to represent the absence of a primary support network.

In [ ]:
# --- 1. FEATURE ENGINEERIN 'personas_a_cargo' (PER) ---

col_pc = 'personas_a_cargo'
new_col_pc = 'personas_cargo_grouped'

# 1.1. Grouping
# consolidate categories with low frequency ('otros_as') and the lack of ('no')
# en una categoría funcional para el modelo ('NO_OTROS').
df[new_col_pc] = df[col_pc].replace({
    'no': 'NO_OTROS',
    'otros_as': 'NO_OTROS'
})

# 1.2. Apply OHE
df = pd.get_dummies(df, columns=[new_col_pc], prefix='PER', drop_first=False)


# --- 2. FEATURE ENGINEERING 'red_vincular' (RV) ---

col_rv = 'red_vincular'
new_col_rv = 'red_vincular_grouped'

# 2.1. Rename and regroup
# consolidate categories with low frequency ('vecinas_os') and the lack of ('no') en 'NO_OTROS'.
df[new_col_rv] = df[col_rv].replace({
    'no': 'NO_OTROS',
    'vecinas_os': 'NO_OTROS',
    'amigas_os': 'AMIGOS' # reaname FRIENDS (AMIGOS)
})

# 2.2. Apply OHE
df = pd.get_dummies(df, columns=[new_col_rv], prefix='RV', drop_first=False)


# --- 4. VERIFICATION ---
print("--- VValidation of Engineered Features ---")

# PERSONAS_A_CARGO
pc_cols = [col for col in df.columns if col.startswith('PER_')]
print("\nGrouped Dependents Count (personas_a_cargo) (PER):")
print(df[pc_cols].sum().to_frame(name='Count').to_markdown())

# red_vincular
rv_cols = [col for col in df.columns if col.startswith('RV_')]
print("\nGrouped Support Network Count (red_vincular)(RV):")
print(df[rv_cols].sum().to_frame(name='Count').to_markdown())

--- Verificación de la Ingeniería de Características ---

Conteo de Personas a Cargo Agrupadas (PER):
|                 |   Conteo |
|:----------------|---------:|
| PER_NO_OTROS    |       32 |
| PER_desconocido |       52 |
| PER_hija_hijo   |      151 |

Conteo de Red Vincular Agrupada (RV):
|                              |   Conteo |
|:-----------------------------|---------:|
| RV_AMIGOS                    |       61 |
| RV_NO_OTROS                  |       12 |
| RV_desconocido               |       62 |
| RV_parientes_convivientes    |       45 |
| RV_parientes_no_convivientes |       55 |


In [ ]:
# Drop original columns
df = df.drop(columns=[col_pc, col_rv], errors='ignore')

### Target and Report Identifiers

The variables medidas_de_proteccion (protection orders) and fecha_fin_de_vigencia (expiration date) were used to define the "ground truth" for the denuncio column. Including them as predictors would allow the model to use them as shortcuts, artificially inflating the metrics. Drop them to prevent Data Leakage.

'Denuncio' is binarized as denuncio_target and converted into the model's numerical target:
* 'NO' $\rightarrow 1$ (Alert)
* 'YES' $\rightarrow 0$ (Baseline)

In [ ]:
# FEATURE ENGINEERING (prevent Data Leakage) ---

col_medida = 'medidas_de_proteccion'
col_fecha = 'fecha_fin_de_vigencia'

# drop columns 
df = df.drop(columns=[col_medida, col_fecha], errors='ignore')

In [ ]:
# verificate the elimination of possible data leakeage columns
if col_medida not in df.columns and col_fecha not in df.columns:
    print("\n Data leakage variables (medidas and fecha) were successfully removed.")
else:
    print("\n Warning: Potential leakage indicators detected in the current feature set. Re-evaluate the data cleaning process.")


 Las variables de fuga de datos (medidas y fecha) fueron eliminadas con éxito.


In [ ]:
#Target Feature Engineering: Binary Encoding

col_denuncio = 'denuncio'

# Creating the numerical target variable: denuncio_target
# NO Denuncia = 1 (Positive Class (Risk/Alert))
df['denuncio_target'] = df[col_denuncio].replace({'no': 1, 'si': 0})

/tmp/ipython-input-2296897140.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['denuncio_target'] = df[col_denuncio].replace({'no': 1, 'si': 0})


In [ ]:
# drop original columns
df = df.drop(columns=[col_denuncio], errors='ignore')

In [ ]:
# verification

print("--- Final Validation of the Engineered Feature Set ---")
print(f"Remaining columns: {df.shape[1]}")

# Verificar el target numérico
print("\nFinal Target Variable Count (denuncio_target):")
print(df['denuncio_target'].value_counts().to_frame(name='Count').to_markdown())

--- Verificación Final de la Ingeniería de Características ---
Total de columnas restantes: 68

Conteo Final del Target Numérico ('denuncio_target'):
|   denuncio_target |   Conteo |
|------------------:|---------:|
|                 0 |      140 |
|                 1 |       95 |


### Boolean to Integer Transformation (0/1 Encoding)

In [ ]:
# 1. Select boolean columns (bool)
boolean_cols = df.select_dtypes(include=['bool']).columns

# 2. Convert to interger (int)
# True is  converted to 1, and False to 0.
df[boolean_cols] = df[boolean_cols].astype(int)

### Post-Preprocessing Structural Audit of the Dataset

In [ ]:
df.head(10)

,fisica,psicologica,sexual,economica,simbolica,ambiental,politica,digital,INGR_CANAL_ASISTENCIAL,INGR_CANAL_ESPONTANEO,...,SUMA_VIOLENCIA,PER_NO_OTROS,PER_desconocido,PER_hija_hijo,RV_AMIGOS,RV_NO_OTROS,RV_desconocido,RV_parientes_convivientes,RV_parientes_no_convivientes,denuncio_target
0,1,1,0,0,0,0,0,0,0,0,...,2,1,0,0,0,0,0,1,0,1
1,1,1,0,1,0,1,0,0,0,0,...,4,0,0,1,0,0,0,0,1,0
2,1,1,0,0,0,0,0,0,0,0,...,2,1,0,0,0,0,0,0,1,1
3,1,1,0,0,0,0,0,0,0,0,...,2,0,1,0,0,0,0,0,1,1
4,1,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,0,1,1
5,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,1
6,0,1,0,1,0,0,0,0,0,0,...,2,0,0,1,0,0,0,0,1,0
7,1,0,0,1,0,0,0,0,0,0,...,2,0,0,1,0,0,0,0,1,0
8,1,1,1,0,1,0,0,0,0,0,...,4,0,0,1,0,0,0,0,1,0
9,1,1,1,1,0,0,0,0,0,0,...,4,0,0,1,0,0,0,0,1,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 68 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   fisica                        235 non-null    int64
 1   psicologica                   235 non-null    int64
 2   sexual                        235 non-null    int64
 3   economica                     235 non-null    int64
 4   simbolica                     235 non-null    int64
 5   ambiental                     235 non-null    int64
 6   politica                      235 non-null    int64
 7   digital                       235 non-null    int64
 8   INGR_CANAL_ASISTENCIAL        235 non-null    int64
 9   INGR_CANAL_ESPONTANEO         235 non-null    int64
 10  INGR_CANAL_JUDICIAL           235 non-null    int64
 11  INGR_CANAL_TERCERO            235 non-null    int64
 12  INGR_DESCONOCIDO              235 non-null    int64
 13  NAC_ARGENTINA                 235 n

# Final Data Export: processed_data.csv

In [ ]:
# 1. Export Processed Data to CSV
df.to_csv('dataset_transformado_ok.csv', index=False)

print("File dataset_transformado_ok.csv successfully generated in the Colab environment.")

# 2. Download File:
from google.colab import files
files.download('dataset_transformado_ok.csv')

print("Download initiated to your computer.")

Archivo 'dataset_transformado_ok.csv' generado con éxito en tu entorno de Colab.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada a tu computadora.
